<a href="https://colab.research.google.com/github/maryamyazdi/Sentiment-Analysis/blob/main/Copy_of_new5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hazm
import hazm as hz
import re
import pickle
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

is_gpu_enabled = torch.cuda.is_available()
device = torch.device("cuda" if is_gpu_enabled else 'cpu')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
df = pd.read_csv('/content/drive/My Drive/train.csv', sep='\t', index_col=0)
train_data = (df.head(30000)).drop(axis=1, columns='label')

def fixup(x):
    x = x.replace('\u200c', '').replace('\xa0','').replace('\r\n',' ').replace('|',' ')
    return x

normalizer = hz.Normalizer()
def my_tokenizer(text):
  text = re.sub(r"[\{\}\؛\*\=\-\+\/\n\(\)]"," ",str(text))
  text = re.sub("[ ]+"," ",text)
  text = re.sub("\!+","!",text)
  text = re.sub("[؟]+","؟",text)
  text = re.sub("\?+","?",text)
  text = re.sub("[.]+",".",text)
  text = fixup(normalizer.normalize(text))
  sentences = hz.sent_tokenize(text)
  words = []
  for every_sentence in sentences:
    words += hz.word_tokenize(every_sentence)
  return words

train_data['comment'] = train_data['comment'].apply(my_tokenizer)

In [3]:
  # print statistics
num_words = train_data['comment'].apply(len)

print('Min length =', num_words.min())
print('Max length =', num_words.max())
print('Mean = {:.2f}'.format(num_words.mean()))
print('Std  = {:.2f}'.format(num_words.std()))
print('mean + 2 * sigma = {:.2f}'.format(num_words.mean() + 2.0 * num_words.std()))

Min length = 2
Max length = 224
Mean = 19.42
Std  = 16.48
mean + 2 * sigma = 52.39


In [4]:
train_data

,comment,label_id
0,"[واقعا, حیف, وقت, که, بنویسم, سرویس, دهیتون, ش...",1
1,"[قرار, بود, ۱, ساعته, برسه, ولی, نیم, ساعت, زو...",0
2,"[قیمت, این, مدل, اصلا, با, کیفیتش, سازگاری, ند...",1
3,"[عالللی, بود, همه, چه, درست, و, به, اندازه, و,...",0
4,"[شیرینی, وانیلی, فقط, یک, مدل, بود, .]",0
...,...,...
29995,"[بسیار, دیر, فرستاده_شد, ., برخرد, پیک, مناسب,...",1
29996,"[پیتزا, پپرونی, آمریکایی, خوش, مزه, و, تند, بو...",1
29997,"[اصلا, مزه, این, کیک, با, کیکی, که, من, قبلا, ...",1
29998,"[غذا, خیلی, سرد, بود, !]",1


In [5]:
max_len = 53
PAD = '<pad>'

def padding_and_trimming(tokens):
  if len(tokens) < max_len:
      num_pads = max_len - len(tokens)
      tokens = [PAD] * num_pads + tokens
  elif len(tokens) > max_len:
      tokens = tokens[:max_len]
  return tokens

In [6]:
train_data['comment'] = train_data['comment'].apply(padding_and_trimming)

In [9]:
!wget 'http://vectors.nlpl.eu/repository/20/61.zip' -O '/content/drive/MyDrive/w2vec.zip'

--2022-01-29 00:08:52--  http://vectors.nlpl.eu/repository/20/61.zip
Resolving vectors.nlpl.eu (vectors.nlpl.eu)... 129.240.189.181
Connecting to vectors.nlpl.eu (vectors.nlpl.eu)|129.240.189.181|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 730416332 (697M) [application/zip]
Saving to: ‘/content/drive/MyDrive/w2vec.zip’

tent/drive/MyDrive/   7%[>                   ]  53.77M  16.0MB/s    eta 42s    ^C


In [ ]:
!unzip /content/drive/MyDrive/w2vec.zip -d /content/drive/MyDrive/w2vec

In [7]:
import gensim
w2v_model = gensim.models.KeyedVectors.load_word2vec_format('/content/drive/MyDrive/w2vec/model.txt', binary=False, unicode_errors='replace')
w2v_weights = torch.FloatTensor(w2v_model.vectors)

In [8]:
import torch.nn as nn


class LSTMClassifier(nn.Module):
  def __init__(self, hidden_size, batch_size, layers_count):
    super(LSTMClassifier, self).__init__()
    self.hidden_size = hidden_size
    self.batch_size = batch_size
    self.layers_count = layers_count

    self.embedding = nn.Embedding.from_pretrained(w2v_weights)
    self.lstm = nn.LSTM(100, hidden_size, layers_count, bidirectional=True, batch_first=True)
    self.classifier_layer = nn.Sequential(
        nn.Linear(2*hidden_size, 100),
        nn.ReLU(),
        nn.Linear(100, 2)
    )
    self.hidden = self.init_hidden()

  def init_hidden(self):
    h = torch.autograd.Variable(torch.zeros((2*self.layers_count, self.batch_size, self.hidden_size)).to(device))
    c = torch.autograd.Variable(torch.zeros((2*self.layers_count, self.batch_size, self.hidden_size)).to(device))
    return h, c

  def forward(self, x):
    x = self.embedding(x)
    x, self.hidden = self.lstm(x, self.hidden)
    x = x.permute(1, 0, 2).detach()
    x = self.classifier_layer(x[-1])
    return x


In [13]:
BATCH_SIZE = 128
lstm_model = LSTMClassifier(hidden_size=512, batch_size=BATCH_SIZE, layers_count=1)

if is_gpu_enabled:
  lstm_model = lstm_model.cuda()

In [14]:
criterion = nn.CrossEntropyLoss()
if is_gpu_enabled:
  criterion = criterion.cuda()

# optimizer = torch.optim.SGD(lstm_model.parameters(), lr=0.001, momentum=0.9)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01, betas=(0.7, 0.99))
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.975)

In [15]:
PAD = '<pad>'
UNK = '<unk>'

class TDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.X_train = dataset['comment']
        self.y_train = dataset['label_id']

    def __len__(self):
        return len(self.X_train)

    def __getitem__(self, index):
        vectors = []
        for token in self.X_train[index]:
          if token == PAD:
            vectors.append(1)
            continue
          try:
            vectors.append(w2v_model.vocab[token].index)
          except KeyError:
            vectors.append(2)
        return torch.tensor(vectors), torch.tensor(self.y_train[index])


dataset = TDataset(train_data)

In [ ]:
train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

lstm_model.train()

losses = []
for epoch in range(5):
  total_loss = 0
  for i, (inputs, targets) in enumerate(train_dataloader):
    optimizer.zero_grad()

    inputs = inputs.to(device)
    targets = targets.to(device)
    outputs = lstm_model(inputs)
    
    loss = criterion(outputs, targets)
    loss.backward()
    scheduler.step()
    total_loss += loss.item()

    print(f'Epoch {epoch + 1}/5 : step {i + 1}/{len(dataset) // BATCH_SIZE}, loss: {loss.item()}')

/usr/local/lib/python3.7/dist-packages/torch/optim/lr_scheduler.py:134: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  "https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate", UserWarning)


Epoch 1/5 : step 1/234, loss: 0.693241536617279
Epoch 1/5 : step 2/234, loss: 0.6962292194366455
Epoch 1/5 : step 3/234, loss: 0.695153534412384
Epoch 1/5 : step 4/234, loss: 0.6941439509391785
Epoch 1/5 : step 5/234, loss: 0.6918800473213196
Epoch 1/5 : step 6/234, loss: 0.6939287185668945
Epoch 1/5 : step 7/234, loss: 0.6966249942779541
Epoch 1/5 : step 8/234, loss: 0.6935563683509827
Epoch 1/5 : step 9/234, loss: 0.6906837224960327
Epoch 1/5 : step 10/234, loss: 0.6959022283554077
Epoch 1/5 : step 11/234, loss: 0.693507730960846
Epoch 1/5 : step 12/234, loss: 0.6974323987960815
Epoch 1/5 : step 13/234, loss: 0.6950230598449707
Epoch 1/5 : step 14/234, loss: 0.6912662386894226
Epoch 1/5 : step 15/234, loss: 0.6960283517837524
Epoch 1/5 : step 16/234, loss: 0.6946023106575012
Epoch 1/5 : step 17/234, loss: 0.6927964687347412
Epoch 1/5 : step 18/234, loss: 0.6965835094451904
Epoch 1/5 : step 19/234, loss: 0.6962357759475708
Epoch 1/5 : step 20/234, loss: 0.6926968693733215
Epoch 1/5 : 